# Instruction-Tuned Model Comparison

A standalone experiment: evaluates 4 instruction-tuned models, each with a
directly comparable BASE model already in the main results, using BOTH
log-probability scoring (all 600 pairs) and prompting (150 sampled pairs).

| Base (already evaluated) | Instruct counterpart |
|---|---|
| Qwen2.5-7B | Qwen2.5-7B-Instruct |
| Llama-3.1-8B | Llama-3.1-8B-Instruct |
| Meltemi-7B-v1 | Meltemi-7B-Instruct-v1 |
| Llama-Krikri-8B-Base | Llama-Krikri-8B-Instruct |

Unlike the base-model prompting comparison (which showed near-chance
prompting accuracy due to lack of instruction-following), this experiment
provides a clean, controlled comparison: same architecture, same size,
only instruction-tuning differs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")

## Config

In [ ]:
INSTRUCT_MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "ilsp/Meltemi-7B-Instruct-v1",
    "ilsp/Llama-Krikri-8B-Instruct",
]

PHENOMENA_DIR = Path("/content/drive/MyDrive/Thesis/data/phenomena")
LOGPROB_OUTPUT_DIR = Path("/content/drive/MyDrive/Thesis/results/autoregressive_instruct")
PROMPTING_OUTPUT_DIR = Path("/content/drive/MyDrive/Thesis/results/prompting_instruct")
LOGPROB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROMPTING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_PAIRS_PER_PHENOMENON_PROMPTING = 25
RANDOM_SEED = 42
SAMPLE_SIZES = [25, 50, 75, 100]
N_REPEATS = 5

print("Config set")

## HuggingFace login (needed for gated Llama models)

In [ ]:
from huggingface_hub import login
login()

## Data loading

In [ ]:
def load_phenomenon_pairs(phenomenon_path):
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"
    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()
    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()
    return list(zip(gram_sentences, ungram_sentences))


def load_all_phenomena(data_dir):
    all_pairs = {}
    for folder in Path(data_dir).iterdir():
        if folder.is_dir():
            name = folder.name[5:] if folder.name.startswith("DONE_") else folder.name
            try:
                pairs = load_phenomenon_pairs(folder)
                all_pairs[name] = pairs
                print(f"  Loaded {name}: {len(pairs)} pairs")
            except FileNotFoundError:
                print(f"  Skipped {folder.name}: files not found")
    return all_pairs


all_data = load_all_phenomena(PHENOMENA_DIR)
print(f"\nTotal phenomena loaded: {len(all_data)}")

## Log-probability scoring (same method as the base-model evaluation)

In [ ]:
class CausalLMScorer:
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def score_sentence(self, sentence):
        sentence = sentence.lower()
        input_ids = self.tokenizer.encode(sentence, return_tensors='pt')
        input_ids = input_ids.to(self.model.device)
        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            avg_log_prob = -outputs.loss.item()
        num_tokens = input_ids.shape[1]
        return {'avg_log_prob': avg_log_prob, 'avg_surprisal': -avg_log_prob, 'num_tokens': num_tokens}


def full_evaluation(scorer, all_data):
    all_results = {}
    for phenomenon_name, pairs in all_data.items():
        correct_count = 0
        pair_results = []
        for gram, ungram in pairs:
            gram_result = scorer.score_sentence(gram)
            ungram_result = scorer.score_sentence(ungram)
            is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']
            if is_correct:
                correct_count += 1
            pair_results.append({
                'grammatical': gram.strip(),
                'ungrammatical': ungram.strip(),
                'gram_avg_log_prob': gram_result['avg_log_prob'],
                'ungram_avg_log_prob': ungram_result['avg_log_prob'],
                'gram_surprisal': gram_result['avg_surprisal'],
                'ungram_surprisal': ungram_result['avg_surprisal'],
                'correct': is_correct
            })
        total_count = len(pairs)
        accuracy = correct_count / total_count
        all_results[phenomenon_name] = {
            'correct': correct_count, 'total': total_count, 'accuracy': accuracy,
            'avg_gram_log_prob': sum(p['gram_avg_log_prob'] for p in pair_results) / total_count,
            'avg_ungram_log_prob': sum(p['ungram_avg_log_prob'] for p in pair_results) / total_count,
            'avg_gram_surprisal': sum(p['gram_surprisal'] for p in pair_results) / total_count,
            'avg_ungram_surprisal': sum(p['ungram_surprisal'] for p in pair_results) / total_count,
            'pairs': pair_results
        }
        print(f"  {phenomenon_name}: {correct_count}/{total_count} = {accuracy:.2%}")
    return all_results


def sample_size_stability(scorer, all_data, sample_sizes=SAMPLE_SIZES, n_repeats=N_REPEATS):
    results = {}
    for phenomenon, pairs in all_data.items():
        pair_correctness = []
        for gram, ungram in pairs:
            g = scorer.score_sentence(gram)
            u = scorer.score_sentence(ungram)
            pair_correctness.append(g['avg_log_prob'] > u['avg_log_prob'])
        phen_results = {}
        for size in sample_sizes:
            if size > len(pair_correctness):
                continue
            accs = []
            for rep in range(n_repeats):
                random.seed(rep)
                sample = random.sample(pair_correctness, size)
                accs.append(sum(sample) / size)
            phen_results[size] = {
                'mean_accuracy': sum(accs) / len(accs),
                'min_accuracy': min(accs), 'max_accuracy': max(accs), 'all_runs': accs
            }
        results[phenomenon] = phen_results
    return results

## Prompting (uses chat template, since these ARE instruction-tuned models)

In [ ]:
def ask_prompting(tokenizer, model, sentence_a, sentence_b):
    prompt = (
        f"Which sentence is grammatically correct Modern Greek?\n"
        f"A: {sentence_a}\n"
        f"B: {sentence_b}\n"
        f"Answer with only the letter A or B."
    )
    messages = [{"role": "user", "content": prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids.to(model.device)
    elif isinstance(encoded, dict):
        input_ids = encoded["input_ids"].to(model.device)
    else:
        input_ids = encoded.to(model.device)

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=10, do_sample=False)

    generated = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    for char in generated.upper():
        if char in ('A', 'B'):
            return char
    return None


def get_sample_pairs(all_data, n_per_phenomenon, seed):
    random.seed(seed)
    sampled = {}
    for phenomenon, pairs in all_data.items():
        sampled[phenomenon] = random.sample(pairs, min(n_per_phenomenon, len(pairs)))
    return sampled


def prompting_evaluation(tokenizer, model, sample_pairs, logprob_pair_results):
    """Runs prompting on the sampled pairs, and looks up the matching
    log-prob verdict (from this same run's full_evaluation results) for
    each pair to compute agreement."""
    comparison_rows = []
    random.seed(RANDOM_SEED)

    for phenomenon, pairs in sample_pairs.items():
        print(f"  {phenomenon} ({len(pairs)} pairs)")
        logprob_lookup = {
            (p['grammatical'].strip(), p['ungrammatical'].strip()): p['correct']
            for p in logprob_pair_results[phenomenon]['pairs']
        }
        for gram, ungram in pairs:
            if random.random() < 0.5:
                sentence_a, sentence_b, correct_letter = gram, ungram, 'A'
            else:
                sentence_a, sentence_b, correct_letter = ungram, gram, 'B'

            answer = ask_prompting(tokenizer, model, sentence_a, sentence_b)
            prompting_correct = (answer == correct_letter)
            logprob_correct = logprob_lookup.get((gram.strip(), ungram.strip()))

            comparison_rows.append({
                'phenomenon': phenomenon,
                'grammatical': gram.strip(), 'ungrammatical': ungram.strip(),
                'model_answer': answer,
                'prompting_correct': prompting_correct,
                'logprob_correct': logprob_correct,
                'methods_agree': (logprob_correct == prompting_correct) if logprob_correct is not None else None,
            })

    return comparison_rows

## Main loop: for each instruct model, run log-prob eval + stability + prompting

In [ ]:
def evaluate_instruct_model(model_name):
    safe_name = model_name.replace('/', '_').replace('.', '_')
    logprob_path = LOGPROB_OUTPUT_DIR / f"{safe_name}_results.json"
    prompting_path = PROMPTING_OUTPUT_DIR / f"{safe_name}_prompting_comparison.json"

    if logprob_path.exists() and prompting_path.exists():
        print(f"Skipping {model_name}: already done.")
        return

    print(f"\n{'='*70}\nEvaluating: {model_name}\n{'='*70}")

    tokenizer, model = None, None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map='auto'
        )
        model.eval()

        scorer = CausalLMScorer(tokenizer, model)

        # 1. Full log-probability evaluation (all 600 pairs)
        if not logprob_path.exists():
            print("\nRunning full log-probability evaluation...")
            full_results = full_evaluation(scorer, all_data)

            print("\nRunning sample-size stability...")
            stability_results = sample_size_stability(scorer, all_data)

            total_correct = sum(r['correct'] for r in full_results.values())
            total_pairs = sum(r['total'] for r in full_results.values())

            logprob_output = {
                'model': model_name,
                'model_type': 'causal_lm_instruct',
                'overall': {
                    'total_pairs': total_pairs, 'total_correct': total_correct,
                    'accuracy': total_correct / total_pairs if total_pairs else 0
                },
                'per_phenomenon': full_results,
                'sample_size_stability': stability_results
            }
            with open(logprob_path, 'w', encoding='utf-8') as f:
                json.dump(logprob_output, f, ensure_ascii=False, indent=2)
                f.flush()
                os.fsync(f.fileno())
            print(f"Saved log-prob results to {logprob_path}")
        else:
            print("Log-prob results already exist, loading for prompting comparison...")
            with open(logprob_path, encoding='utf-8') as f:
                logprob_output = json.load(f)
            full_results = logprob_output['per_phenomenon']

        # 2. Prompting evaluation (25 sampled pairs per phenomenon)
        if not prompting_path.exists():
            print("\nRunning prompting evaluation...")
            sample_pairs = get_sample_pairs(all_data, N_PAIRS_PER_PHENOMENON_PROMPTING, RANDOM_SEED)
            comparison_rows = prompting_evaluation(tokenizer, model, sample_pairs, full_results)

            prompting_acc = sum(r['prompting_correct'] for r in comparison_rows) / len(comparison_rows)
            rows_with_logprob = [r for r in comparison_rows if r['logprob_correct'] is not None]
            logprob_acc_sampled = (sum(r['logprob_correct'] for r in rows_with_logprob) / len(rows_with_logprob)
                                    if rows_with_logprob else None)
            agreement = (sum(r['methods_agree'] for r in rows_with_logprob) / len(rows_with_logprob)
                         if rows_with_logprob else None)

            print(f"\nPrompting accuracy: {prompting_acc:.1%}")
            print(f"Log-prob accuracy (sampled pairs): {logprob_acc_sampled:.1%}")
            print(f"Method agreement: {agreement:.1%}")

            prompting_output = {
                'model': model_name,
                'backend': 'local',
                'n_pairs_per_phenomenon': N_PAIRS_PER_PHENOMENON_PROMPTING,
                'prompting_accuracy': prompting_acc,
                'logprob_accuracy': logprob_acc_sampled,
                'method_agreement': agreement,
                'pairs': comparison_rows,
            }
            with open(prompting_path, 'w', encoding='utf-8') as f:
                json.dump(prompting_output, f, ensure_ascii=False, indent=2)
                f.flush()
                os.fsync(f.fileno())
            print(f"Saved prompting results to {prompting_path}")

    except Exception as e:
        import traceback
        print(f"FAILED on {model_name}: {type(e).__name__}: {e}")
        traceback.print_exc()

    finally:
        try:
            del model, tokenizer
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()

## Run for all 4 instruct models

In [9]:
for model_name in INSTRUCT_MODELS:
    evaluate_instruct_model(model_name)

print("\n" + "="*70)
print("ALL INSTRUCT MODELS DONE")
print("="*70)


Evaluating: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Running full log-probability evaluation...
FAILED on ilsp/Llama-Krikri-8B-Instruct: OutOfMemoryError: CUDA out of memory. Tried to allocate 1.14 GiB. GPU 0 has a total capacity of 14.56 GiB of which 273.81 MiB is free. Including non-PyTorch memory, this process has 14.29 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 33.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


Traceback (most recent call last):
  File "/tmp/ipykernel_824/42930891.py", line 25, in evaluate_instruct_model
    full_results = full_evaluation(scorer, all_data)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_824/4051410073.py", line 23, in full_evaluation
    gram_result = scorer.score_sentence(gram)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_824/4051410073.py", line 11, in score_sentence
    outputs = self.model(input_ids, labels=input_ids)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/


ALL INSTRUCT MODELS DONE


## Summary: base vs. instruct comparison

In [10]:
import pandas as pd

BASE_TO_INSTRUCT = {
    "Qwen/Qwen2.5-7B": "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.1-8B": "meta-llama/Llama-3.1-8B-Instruct",
    "ilsp/Meltemi-7B-v1": "ilsp/Meltemi-7B-Instruct-v1",
    "ilsp/Llama-Krikri-8B-Base": "ilsp/Llama-Krikri-8B-Instruct",
}

BASE_LOGPROB_DIR = Path("/content/drive/MyDrive/Thesis/results/autoregressive")
BASE_PROMPTING_DIR = Path("/content/drive/MyDrive/Thesis/results/prompting")

summary_rows = []
for base_name, instruct_name in BASE_TO_INSTRUCT.items():
    base_safe = base_name.replace('/', '_').replace('.', '_')
    instruct_safe = instruct_name.replace('/', '_').replace('.', '_')

    row = {'Base Model': base_name.split('/')[-1], 'Instruct Model': instruct_name.split('/')[-1]}

    # Base log-prob accuracy
    base_logprob_file = BASE_LOGPROB_DIR / f"{base_safe}_results.json"
    if base_logprob_file.exists():
        with open(base_logprob_file, encoding='utf-8') as f:
            row['Base Log-prob Acc.'] = json.load(f)['overall']['accuracy'] * 100

    # Instruct log-prob accuracy
    instruct_logprob_file = LOGPROB_OUTPUT_DIR / f"{instruct_safe}_results.json"
    if instruct_logprob_file.exists():
        with open(instruct_logprob_file, encoding='utf-8') as f:
            row['Instruct Log-prob Acc.'] = json.load(f)['overall']['accuracy'] * 100

    # Base prompting accuracy
    base_prompting_file = BASE_PROMPTING_DIR / f"{base_safe}_prompting_comparison.json"
    if base_prompting_file.exists():
        with open(base_prompting_file, encoding='utf-8') as f:
            row['Base Prompting Acc.'] = json.load(f)['prompting_accuracy'] * 100

    # Instruct prompting accuracy
    instruct_prompting_file = PROMPTING_OUTPUT_DIR / f"{instruct_safe}_prompting_comparison.json"
    if instruct_prompting_file.exists():
        with open(instruct_prompting_file, encoding='utf-8') as f:
            row['Instruct Prompting Acc.'] = json.load(f)['prompting_accuracy'] * 100

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(1).to_string(index=False))
summary_df.to_csv(LOGPROB_OUTPUT_DIR.parent / "base_vs_instruct_summary.csv", index=False)
print(f"\nSaved to {LOGPROB_OUTPUT_DIR.parent / 'base_vs_instruct_summary.csv'}")

          Base Model           Instruct Model  Instruct Log-prob Acc.  Base Prompting Acc.  Instruct Prompting Acc.  Base Log-prob Acc.
          Qwen2.5-7B      Qwen2.5-7B-Instruct                    91.7                 75.3                     89.3                 NaN
        Llama-3.1-8B    Llama-3.1-8B-Instruct                     NaN                 70.0                      NaN                 NaN
       Meltemi-7B-v1   Meltemi-7B-Instruct-v1                    90.8                 49.3                     48.7                92.0
Llama-Krikri-8B-Base Llama-Krikri-8B-Instruct                     NaN                 40.7                      NaN                88.5

Saved to /content/drive/MyDrive/Thesis/results/base_vs_instruct_summary.csv
